In [1]:
# ============================================================
# ArcGIS Pro (ArcPy) — Centerlines por año (2017–2022) desde LCP/EPOF
# Flujo por año:
# 1) LineDensity ponderada por "conf"
# 2) Slice (10 cuantiles) -> sliced_conf_weighted
# 3) Mask >=2 (quita gris)
# 4) Thin (skeleton) -> eje
# 5) RasterToPolyline -> centerline_full
# 6) AddSurfaceInformation sobre sliced -> Z_MIN/Z_MAX/Z_MEAN
# 7) LEVEL_DOM = round(Z_MAX)  (nivel dominante 2..10)
#
# INPUT (por año):
#   ...\rutas_LCP_EPOF_{YEAR}_conf25.shp
# OUTPUT (por año):
#   ...\center_lines\{YEAR}\density_conf_weighted.tif
#   ...\center_lines\{YEAR}\sliced_conf_weighted.tif
#   ...\center_lines\{YEAR}\mask_ge2.tif
#   ...\center_lines\{YEAR}\thin_full.tif
#   ...\center_lines\{YEAR}\centerline_full.shp
# ============================================================

import os
import arcpy
from arcpy.sa import LineDensity, Slice, Con, Thin

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# -----------------------
# Config
# -----------------------
base_in = r"C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output"
out_root = r"C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines"
os.makedirs(out_root, exist_ok=True)

years = list(range(2017, 2023))  # 2017..2022 (incluye 2022)
conf_field = "conf"

# Parámetros (en unidades del CRS del shapefile; idealmente metros)
cell_size = 200
search_radius = 2000

# Slice (para gradientes)
n_classes = 10
slice_method = "EQUAL_AREA"   # cuantiles

# Mask (quita gris)
mask_min_class = 2  # conservar >=2

# Thin
thin_method = "NO_FILTER"

def safe_calc_stats(ras_path):
    try:
        arcpy.management.CalculateStatistics(ras_path)
    except:
        pass

for yr in years:
    print(f"\n================= {yr} =================")

    in_fc = os.path.join(base_in, f"rutas_LCP_EPOF_{yr}_conf25.shp")
    if not arcpy.Exists(in_fc):
        print(f"NO EXISTE -> {in_fc}  (saltando)")
        continue

    # carpeta por año
    out_dir = os.path.join(out_root, str(yr))
    os.makedirs(out_dir, exist_ok=True)

    dens_ras  = os.path.join(out_dir, "density_conf_weighted.tif")
    slice_ras = os.path.join(out_dir, "sliced_conf_weighted.tif")
    mask_ras  = os.path.join(out_dir, "mask_ge2.tif")
    thin_ras  = os.path.join(out_dir, "thin_full.tif")
    centerline = os.path.join(out_dir, "centerline_full.shp")

    # --- validar campo conf ---
    fields = [f.name for f in arcpy.ListFields(in_fc)]
    if conf_field not in fields:
        raise ValueError(f"[{yr}] No existe el campo '{conf_field}' en {in_fc}. Campos: {fields}")

    # --- entorno por año ---
    desc = arcpy.Describe(in_fc)
    arcpy.env.outputCoordinateSystem = desc.spatialReference
    arcpy.env.extent = desc.extent

    # 1) Density ponderada por conf
    density = LineDensity(
        in_polyline_features=in_fc,
        population_field=conf_field,
        cell_size=cell_size,
        search_radius=search_radius,
        area_unit_scale_factor="SQUARE_KILOMETERS"
    )
    density.save(dens_ras)
    safe_calc_stats(dens_ras)

    # 2) Slice (cuantiles)
    ras = arcpy.Raster(dens_ras)
    sliced = Slice(ras, n_classes, slice_method)
    sliced.save(slice_ras)
    safe_calc_stats(slice_ras)

    # 3) Mask >=2
    mask = Con(arcpy.Raster(slice_ras) >= mask_min_class, 1)
    mask.save(mask_ras)
    safe_calc_stats(mask_ras)

    # 4) Thin
    thin = Thin(mask, "ZERO", thin_method)
    thin.save(thin_ras)
    safe_calc_stats(thin_ras)

    # 5) Raster -> Polyline
    arcpy.conversion.RasterToPolyline(
        in_raster=thin_ras,
        out_polyline_features=centerline,
        background_value="ZERO",
        simplify="NO_SIMPLIFY"
    )

    # 6) Sample del sliced sobre líneas (Z_MIN/Z_MAX/Z_MEAN)
    # Nota: usa arcpy.ddd (3D Analyst). Si falla por licencia, dime y lo cambiamos.
    arcpy.ddd.AddSurfaceInformation(
        in_feature_class=centerline,
        in_surface=slice_ras,
        out_property="Z_MIN;Z_MAX;Z_MEAN",
        method="BILINEAR"
    )

    # 7) Campo entero dominante
    if "LEVEL_DOM" not in [f.name for f in arcpy.ListFields(centerline)]:
        arcpy.management.AddField(centerline, "LEVEL_DOM", "LONG")
    arcpy.management.CalculateField(centerline, "LEVEL_DOM", "round(!Z_MAX!)", "PYTHON3")

    print("OK ->", centerline)

print("\nLISTO ✅ 2017–2022 procesados")
arcpy.CheckInExtension("Spatial")



================= 2017 =================
OK -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines\2017\centerline_full.shp

================= 2018 =================
OK -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines\2018\centerline_full.shp

================= 2019 =================
OK -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines\2019\centerline_full.shp

================= 2020 =================
OK -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines\2020\centerline_full.shp

================= 2021 =================
OK -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_p

'CheckedOut'

In [1]:
# ============================================================
# ArcGIS Pro (ArcPy) — Export FINAL centerlines (2017–2022) a una sola carpeta
# - Toma los centerline_full.shp ya generados en:
#     ...\output\center_lines\{YEAR}\centerline_full.shp
# - Exporta UNO por año (sin subcarpetas) a:
#     ...\output\center_lines_final
# - Nombre:
#     rutas_LCP_EPOF_{YEAR}_conf25_corr.shp
# - Agrega campo long_km (longitud en km)
# ============================================================

import os
import arcpy

arcpy.env.overwriteOutput = True

in_root = r"C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines"
out_dir = r"C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines_final"
os.makedirs(out_dir, exist_ok=True)

years = list(range(2017, 2023))  # 2017..2022

for yr in years:
    in_fc = os.path.join(in_root, str(yr), "centerline_full.shp")
    if not arcpy.Exists(in_fc):
        print(f"[{yr}] NO EXISTE -> {in_fc} (saltando)")
        continue

    out_fc = os.path.join(out_dir, f"rutas_LCP_EPOF_{yr}_conf25_corr.shp")

    # 1) Copiar a carpeta final (sin subcarpetas)
    arcpy.management.CopyFeatures(in_fc, out_fc)

    # 2) Crear campo long_km si no existe
    fields = [f.name.lower() for f in arcpy.ListFields(out_fc)]
    if "long_km" not in fields:
        arcpy.management.AddField(out_fc, "long_km", "DOUBLE")

    # 3) Calcular longitud en km
    # Usa SHAPE@LENGTH (unidades del CRS). Si CRS está en metros -> km = /1000.
    # Si no está en metros, re-proyecta antes de calcular (idealmente todo en 3116).
    sr = arcpy.Describe(out_fc).spatialReference
    unit = getattr(sr, "linearUnitName", "").lower()

    if "meter" in unit or "metre" in unit:
        expr = "!shape.length@meters! / 1000.0"
        arcpy.management.CalculateField(out_fc, "long_km", expr, "PYTHON3")
    else:
        # fallback: igual intenta en meters (ArcGIS suele calcular geodésico cuando aplica)
        # pero lo correcto es proyectar a CRS en metros.
        expr = "!shape.length@kilometers!"
        arcpy.management.CalculateField(out_fc, "long_km", expr, "PYTHON3")

    print(f"OK [{yr}] -> {out_fc}")

print("LISTO ✅ Export final + long_km")


OK [2017] -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines_final\rutas_LCP_EPOF_2017_conf25_corr.shp
OK [2018] -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines_final\rutas_LCP_EPOF_2018_conf25_corr.shp
OK [2019] -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines_final\rutas_LCP_EPOF_2019_conf25_corr.shp
OK [2020] -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines_final\rutas_LCP_EPOF_2020_conf25_corr.shp
OK [2021] -> C:\Users\d.millanorduz\OneDrive - Universidad de los Andes\Diana_CESED\rutas\modeling_routes_CESED\05_post_processing\output\center_lines_final\rutas_LCP_EPOF_2021_conf25_corr.shp
OK [2022] -> C:\Users\d.millanorduz